# Beat Tracker and Tempo Estimation
- Asa Picton, Connor Richardson, Dylan Baecker
- CSC475

### 1. Import libraries

In [1]:

from pathlib import Path
from joblib import Parallel, delayed
import joblib
import numpy as np
import pandas as pd
import librosa as lb
from tqdm.auto import tqdm

import scipy.signal
import matplotlib.pyplot as plt

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor



/Users/connor/miniconda3/envs/music/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Load audio and tempo data

In [2]:
audio_path = Path("giantsteps-tempo-dataset/audio")
label_path = Path("giantsteps-tempo-dataset/annotations/tempo")

# Load tempo labels
tempo_labels = {}
for file in label_path.iterdir():
    tempo_labels[file.stem] = float(file.read_text().strip())

# Load dataset
dataset = []
for file in audio_path.iterdir():
    track_id = file.stem
    if track_id in tempo_labels:
        dataset.append((file, tempo_labels[track_id]))

print(f"Loaded {len(dataset)} audio tracks with tempo labels")


Loaded 664 audio tracks with tempo labels


In [3]:
print(len(dataset))
print(dataset[0])

664
(PosixPath('giantsteps-tempo-dataset/audio/3226172.LOFI.mp3'), 69.0)


### 3. Feature Extraction

In [4]:
def extract_features(y_audio, sr):

    features = []

    # 1. Onset strength envelope
    onset_env = lb.onset.onset_strength(y=y_audio, sr=sr)

    features.extend([
        np.mean(onset_env),
        np.std(onset_env),
        np.max(onset_env)
    ])

    # 2. Librosa tempo estimate
    tempo_est = lb.feature.tempo(onset_envelope=onset_env, sr=sr)[0]
    features.append(tempo_est)

    # 3. Tempogram
    tg = lb.feature.tempogram(onset_envelope=onset_env, sr=sr)
    tempo_freqs = lb.tempo_frequencies(tg.shape[0], sr=sr)

    tg_mean = np.mean(tg, axis=1)

    if tg_mean.max() > 0:
        tg_mean = tg_mean / tg_mean.max()

    peaks, props = scipy.signal.find_peaks(tg_mean, height=0.1)

    if len(peaks) > 0:
        order = np.argsort(props["peak_heights"])[::-1]
        peaks = peaks[order][:5]

        peak_bpms = tempo_freqs[peaks]
        peak_strengths = tg_mean[peaks]
    else:
        peak_bpms = np.zeros(5)
        peak_strengths = np.zeros(5)

    peak_bpms = np.pad(peak_bpms, (0, 5-len(peak_bpms)))
    peak_strengths = np.pad(peak_strengths, (0, 5-len(peak_strengths)))

    features.extend(peak_bpms)
    features.extend(peak_strengths)

    return np.array(features)

In [5]:
def process_track(path, bpm):
    y_audio, sr = lb.load(path, duration=30)
    features = extract_features(y_audio, sr)

    return features, bpm


In [6]:
CACHE_FILE = Path("features_cache.pkl")

if CACHE_FILE.exists():
    print("Loading features from cache")
    X, y = joblib.load(CACHE_FILE)
    print(f"Loaded {len(X)} cached feature vectors.")
else:
    print("Extracting features")

    results = Parallel(n_jobs=-1)(
        delayed(process_track)(path, bpm)
        for path, bpm in tqdm(dataset, desc="Tracks")
    )
    X, y = zip(*results)

    joblib.dump((X, y), CACHE_FILE)
    print(f"Extracted features for {len(X)} tracks. Saved to {CACHE_FILE}.")

Extracting features


Tracks: 100%|██████████| 664/664 [00:45<00:00, 14.65it/s]


Extracted features for 664 tracks. Saved to features_cache.pkl.


In [7]:
# Convert features to numpy arrays
X = np.array(X)  
y = np.array(y) 

print(f"X shape: {X.shape}")  
print(f"y shape: {y.shape}")  

X shape: (664, 14)
y shape: (664,)


### 4. Model Training 

##### 4.1 Random Forest Regressor

In [8]:
def half_double_corrected_mae(y_true, y_pred):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)

    return np.mean(np.min(errors, axis=0))


def acc1(y_true, y_pred, tol=0.04):
    return np.mean(np.abs(y_pred - y_true) <= tol * y_true)


def acc2(y_true, y_pred, tol=0.04):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)

    min_errors = np.min(errors, axis=0)

    return np.mean(min_errors <= tol * y_true)



def training_loop(X, y):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    all_mae = []
    all_acc1 = []
    all_acc2 = []

    def objective(trial, X_train, y_train):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
            "max_depth": trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.1),
        }

        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=3,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )

        return scores.mean()

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        print(f"\nFold {fold+1}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        study = optuna.create_study(direction="maximize")

        study.optimize(
            lambda trial: objective(trial, X_train, y_train),
            n_trials=25,
            show_progress_bar=True
        )

        best_params = study.best_params

        print(f"\nFold {fold+1} best params: {best_params}")
        print(f"Best CV score: {study.best_value:.3f}")

        model = RandomForestRegressor(
            **best_params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

    
    
        errors = np.abs(y_pred - y_test)
        print(f"Errors > 30 BPM: {np.sum(errors > 30)} / {len(errors)}")
        print(f"Errors > 60 BPM: {np.sum(errors > 60)} / {len(errors)}")

        mae = half_double_corrected_mae(y_test, y_pred)
        acc1_score = acc1(y_test, y_pred)
        acc2_score = acc2(y_test, y_pred)

        print("\nFold Results")
        print(f"MAE (half/double corrected): {mae:.2f} BPM")
        print(f"Acc1 (±4% strict): {acc1_score:.3f}")
        print(f"Acc2 (±4% half/double): {acc2_score:.3f}")

        all_mae.append(mae)
        all_acc1.append(acc1_score)
        all_acc2.append(acc2_score)

    print("\nResults")

    print(f"MAE (half/double corrected): {np.mean(all_mae):.2f} ± {np.std(all_mae):.2f} BPM")
    print(f"Acc1 (±4% strict): {np.mean(all_acc1):.3f} ± {np.std(all_acc1):.3f}")
    print(f"Acc2 (±4% half/double): {np.mean(all_acc2):.3f} ± {np.std(all_acc2):.3f}")


training_loop(X, y)


Fold 1


Best trial: 22. Best value: -19.0576: 100%|██████████| 25/25 [00:21<00:00,  1.17it/s]



Fold 1 best params: {'n_estimators': 100, 'max_depth': 27, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.028058963290040234}
Best CV score: -19.058
Errors > 30 BPM: 41 / 133
Errors > 60 BPM: 6 / 133

Fold Results
MAE (half/double corrected): 14.37 BPM
Acc1 (±4% strict): 0.368
Acc2 (±4% half/double): 0.383

Fold 2


Best trial: 22. Best value: -18.728: 100%|██████████| 25/25 [00:21<00:00,  1.18it/s]



Fold 2 best params: {'n_estimators': 200, 'max_depth': 28, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.06286955831450566}
Best CV score: -18.728
Errors > 30 BPM: 31 / 133
Errors > 60 BPM: 9 / 133

Fold Results
MAE (half/double corrected): 12.76 BPM
Acc1 (±4% strict): 0.338
Acc2 (±4% half/double): 0.346

Fold 3


...


Fold 3 best params: {'n_estimators': 450, 'max_depth': 25, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.0985612840878363}
Best CV score: -20.193
Errors > 30 BPM: 22 / 133
Errors > 60 BPM: 7 / 133

Fold Results
MAE (half/double corrected): 11.99 BPM
Acc1 (±4% strict): 0.293
Acc2 (±4% half/double): 0.308

Fold 4


Best trial: 15. Best value: -19.8952: 100%|██████████| 25/25 [00:25<00:00,  1.00s/it]



Fold 4 best params: {'n_estimators': 350, 'max_depth': 23, 'min_samples_split': 15, 'min_samples_leaf': 4, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 9.480179052923782e-05}
Best CV score: -19.895
Errors > 30 BPM: 29 / 133
Errors > 60 BPM: 7 / 133

Fold Results
MAE (half/double corrected): 13.89 BPM
Acc1 (±4% strict): 0.301
Acc2 (±4% half/double): 0.301

Fold 5


Best trial: 20. Best value: -18.6264: 100%|██████████| 25/25 [00:21<00:00,  1.16it/s]



Fold 5 best params: {'n_estimators': 250, 'max_depth': 23, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.04263053918329973}
Best CV score: -18.626
Errors > 30 BPM: 39 / 132
Errors > 60 BPM: 5 / 132

Fold Results
MAE (half/double corrected): 16.85 BPM
Acc1 (±4% strict): 0.235
Acc2 (±4% half/double): 0.242

Results
MAE (half/double corrected): 13.97 ± 1.66 BPM
Acc1 (±4% strict): 0.307 ± 0.045
Acc2 (±4% half/double): 0.316 ± 0.047
